In [6]:
# Install required packages
%pip install onnx onnxruntime pillow numpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 73.2 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 20.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 676.9/676.9 kB 26.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [onnxruntime] [onnxruntime]
Note: you may need to restart the kernel to use updated packages.


Homework

In this homework, we'll deploy the Straight vs Curly Hair Type model we trained in the previous homework.

Download the model files from here:

    https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx.data
    https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle/hair_classifier_v1.onnx

With wget:

PREFIX="https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
DATA_URL="${PREFIX}/hair_classifier_v1.onnx.data"
MODEL_URL="${PREFIX}/hair_classifier_v1.onnx"
wget ${DATA_URL}
wget ${MODEL_URL}

Question 1

To be able to use this model, we need to know the name of the input and output nodes.

What's the name of the output:

    output
    sigmoid
    softmax
    prediction

Preparing the image

You'll need some code for downloading and resizing images. You can use this code:

from io import BytesIO
from urllib import request

from PIL import Image

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

For that, you'll need to have pillow installed:

pip install pillow

Question 2: Target size

Let's download and resize this image:

https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

Based on the previous homework, what should be the target size for the image?

    64x64
    128x128
    200x200
    256x256

Question 3

Now we need to turn the image into numpy array and pre-process it.

    Tip: Check the previous homework. What was the pre-processing we did there?

After the pre-processing, what's the value in the first pixel, the R channel?

    -10.73
    -1.073
    1.073
    10.73

Question 4

Now let's apply this model to this image. What's the output of the model?

    0.09
    0.49
    0.69
    0.89

Prepare the lambda code

Now you need to copy all the code into a separate python file. You will need to use this file for the next two questions.

Tip: you can test this file locally with ipython or Jupyter Notebook by importing the file and invoking the function from this file.
Docker

For the next two questions, we'll use a Docker image that we already prepared. This is the Dockerfile that we used for creating the image:

FROM public.ecr.aws/lambda/python:3.13

COPY hair_classifier_empty.onnx.data .
COPY hair_classifier_empty.onnx .

Note that it uses Python 3.13.

The docker image is published to agrigorev/model-2025-hairstyle:v1.

A few notes:

    The image already contains a model and it's not the same model as the one we used for questions 1-4.

Question 5

Download the base image agrigorev/model-2025-hairstyle:v1. You can do it with docker pull.

So what's the size of this base image?

    88 Mb
    208 Mb
    608 Mb
    1208 Mb

You can get this information when running docker images - it'll be in the "SIZE" column.
Question 6

Now let's extend this docker image, install all the required libraries and add the code for lambda.

You don't need to include the model in the image. It's already included. The name of the file with the model is hair_classifier_empty.onnx and it's in the current workdir in the image (see the Dockerfile above for the reference). The provided model requires the same preprocessing for images regarding target size and rescaling the value range than used in homework 8.

Now run the container locally.

Score this image: https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg

What's the output from the model?

    -1.0
    -0.10
    0.10
    1.0



In [7]:
# Question 1: Find the name of the output node in the ONNX model

import onnx
import numpy as np

# Download the model files
PREFIX = "https://github.com/alexeygrigorev/large-datasets/releases/download/hairstyle"
DATA_URL = f"{PREFIX}/hair_classifier_v1.onnx.data"
MODEL_URL = f"{PREFIX}/hair_classifier_v1.onnx"

import urllib.request
urllib.request.urlretrieve(DATA_URL, "hair_classifier_v1.onnx.data")
urllib.request.urlretrieve(MODEL_URL, "hair_classifier_v1.onnx")

# Load and inspect the ONNX model
model = onnx.load("hair_classifier_v1.onnx")

# Get input and output names
print("Input nodes:")
for input_tensor in model.graph.input:
    print(f"  Name: {input_tensor.name}, Shape: {[dim.dim_value for dim in input_tensor.type.tensor_type.shape.dim]}")

print("\nOutput nodes:")
for output_tensor in model.graph.output:
    print(f"  Name: {output_tensor.name}, Shape: {[dim.dim_value for dim in output_tensor.type.tensor_type.shape.dim]}")

# Answer to Question 1
output_name = model.graph.output[0].name
print(f"\nAnswer to Question 1: {output_name}")


Input nodes:
  Name: input, Shape: [0, 3, 200, 200]

Output nodes:
  Name: output, Shape: [0, 1]

Answer to Question 1: output


In [8]:
# Question 2: Target size and Question 3: Preprocessing

from io import BytesIO
from urllib import request
from PIL import Image
import numpy as np

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

# Download the test image
image_url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
img = download_image(image_url)

# Check the model input shape to determine target size
# The model input shape will tell us the target size
model_shape = [dim.dim_value for dim in model.graph.input[0].type.tensor_type.shape.dim]
# ONNX models typically have shape [batch, channels, height, width] or [batch, height, width, channels]
# Let's check: if shape is [1, 3, H, W] or [1, H, W, 3], we extract H and W
if len(model_shape) == 4:
    if model_shape[1] == 3:  # channels-first: [batch, channels, height, width]
        target_size = (model_shape[2], model_shape[3])
    else:  # channels-last: [batch, height, width, channels]
        target_size = (model_shape[1], model_shape[2])
else:
    # Default to 200x200 based on HW8
    target_size = (200, 200)
    
print(f"Model input shape: {model_shape}")
print(f"Target size determined: {target_size}")
img_resized = prepare_image(img, target_size)

print(f"Answer to Question 2: {target_size[0]}x{target_size[1]}")

# Convert to numpy array and preprocess
# From HW8: normalization with ImageNet mean and std
# Normalization: (pixel / 255.0 - mean) / std
img_array = np.array(img_resized, dtype=np.float32)

# ImageNet normalization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

# Normalize
img_normalized = (img_array / 255.0 - mean) / std

# Convert to channels-first format (C, H, W) for ONNX
img_normalized = img_normalized.transpose(2, 0, 1)

# Get the first pixel, R channel (channel 0)
first_pixel_r = img_normalized[0, 0, 0]
print(f"\nAnswer to Question 3: {first_pixel_r:.3f}")


Model input shape: [0, 3, 200, 200]
Target size determined: (200, 200)
Answer to Question 2: 200x200

Answer to Question 3: -1.073


In [9]:
# Question 4: Apply the model to the image

import onnxruntime as ort

# Prepare input (add batch dimension)
input_data = img_normalized[np.newaxis, :, :, :].astype(np.float32)

# Create inference session
session = ort.InferenceSession("hair_classifier_v1.onnx")

# Get input name
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Run inference
outputs = session.run([output_name], {input_name: input_data})

# The output is a sigmoid (binary classification), so we get a probability
prediction = outputs[0][0][0]

print(f"Answer to Question 4: {prediction:.2f}")


Answer to Question 4: 0.09


In [ ]:
# Question 5: Check Docker image size

import subprocess
import os

# Pull the Docker image (if not already pulled)
print("Pulling Docker image (if needed)...")
result = subprocess.run(['docker', 'pull', 'agrigorev/model-2025-hairstyle:v1'], 
                       capture_output=True, text=True)
if result.stdout:
    print(result.stdout)
if result.stderr and "Error" not in result.stderr:
    print(result.stderr)

# Check image size
print("\nChecking image size...")
result = subprocess.run(['docker', 'images', 'agrigorev/model-2025-hairstyle:v1'], 
                       capture_output=True, text=True)
print(result.stdout)

# Extract size from output
lines = result.stdout.strip().split('\n')
if len(lines) > 1:
    # Header line and data line
    size_line = lines[1].split()
    if len(size_line) >= 7:
        size_str = size_line[6]  # SIZE column
        print(f"\nImage size: {size_str}")
        
        # Parse the size to determine which option it matches
        # Docker shows sizes like "921MB", "1.2GB", etc.
        size_value = size_str.upper()
        if 'GB' in size_value:
            # Convert GB to MB for comparison
            gb_value = float(size_value.replace('GB', ''))
            mb_value = gb_value * 1024
        elif 'MB' in size_value:
            mb_value = float(size_value.replace('MB', ''))
        else:
            mb_value = float(size_value)
        
        print(f"Size in MB: {mb_value:.0f} MB")
        
        # Compare with options: 88, 208, 608, 1208
        options = [88, 208, 608, 1208]
        closest = min(options, key=lambda x: abs(x - mb_value))
        
        print(f"\nAnswer to Question 5: {closest} Mb")
        print(f"(Image size is {mb_value:.0f} MB, closest to {closest} Mb)")
else:
    print("\nCould not find image. Run manually: docker images | grep agrigorev/model-2025-hairstyle")


Pulling Docker image (if needed)...
v1: Pulling from agrigorev/model-2025-hairstyle
Digest: sha256:9e43d5a5323f7f07688c0765d3c0137af66d0154af37833ed721d6b4de6df528
Status: Image is up to date for agrigorev/model-2025-hairstyle:v1
docker.io/agrigorev/model-2025-hairstyle:v1


Checking image size...
REPOSITORY                       TAG       IMAGE ID       CREATED       SIZE
agrigorev/model-2025-hairstyle   v1        9e43d5a5323f   11 days ago   921MB


Image size: 921MB
Size in MB: 921 MB

Answer to Question 5: 1208 Mb
(Image size is 921 MB, closest to 1208 Mb)


In [4]:
# Prepare lambda code for Question 6

# This code will be used in the Docker container
# The model file is already in the image as hair_classifier_empty.onnx

lambda_code = '''
import json
import onnxruntime as ort
import numpy as np
from io import BytesIO
from urllib import request
from PIL import Image

# Load the model (already in the image)
session = ort.InferenceSession("hair_classifier_empty.onnx")
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img

def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

def lambda_handler(event, context):
    url = event['url']
    
    # Download and prepare image
    img = download_image(url)
    img = prepare_image(img, (200, 200))
    
    # Preprocess
    img_array = np.array(img, dtype=np.float32)
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_normalized = (img_array / 255.0 - mean) / std
    img_normalized = img_normalized.transpose(2, 0, 1)
    input_data = img_normalized[np.newaxis, :, :, :].astype(np.float32)
    
    # Run inference
    outputs = session.run([output_name], {input_name: input_data})
    result = float(outputs[0][0][0])
    
    return {
        'statusCode': 200,
        'body': json.dumps({'result': result})
    }
'''

print("Lambda code prepared. Save this to lambda_function.py for Question 6")


Lambda code prepared. Save this to lambda_function.py for Question 6


## Question 6: Docker Setup

To answer Question 6:

1. Build the Docker image:
```bash
docker build -t hair-classifier-lambda .
```

2. Run the container locally:
```bash
docker run -p 9000:8080 hair-classifier-lambda
```

3. In another terminal, test it:
```bash
curl -XPOST "http://localhost:9000/2015-03-31/functions/function/invocations" -d '{"url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'
```

Or use the test script:
```bash
docker run --rm -v $(pwd):/var/task hair-classifier-lambda python test_lambda_local.py
```


## Summary of Answers

**Question 1:** Check the output above from cell 1 (likely "output", "sigmoid", "softmax", or "prediction")

**Question 2:** 200x200 (based on model input shape from HW8)

**Question 3:** Check the output above from cell 2 (should be around -1.073)

**Question 4:** Check the output above from cell 3 (should be a value between 0 and 1)

**Question 5:** Run cell 4 or manually: `docker pull agrigorev/model-2025-hairstyle:v1` then `docker images | grep agrigorev/model-2025-hairstyle`

**Question 6:** Build and run the Docker container as shown above, then test with the image URL


# Question 6: Build and run Docker container, then test
# Note: Make sure Docker Desktop is running before executing this cell

# Install requests if needed
%pip install requests

import subprocess
import json
import time
import requests

# First, check if Docker is running
try:
    result = subprocess.run(['docker', 'ps'], capture_output=True, text=True, timeout=5)
    if result.returncode != 0:
        print("ERROR: Docker daemon is not running!")
        print("Please start Docker Desktop and try again.")
        print("\nOnce Docker is running, execute the following commands manually:")
        print("1. docker build -t hair-classifier-lambda .")
        print("2. docker run -d -p 9000:8080 --name hair-lambda hair-classifier-lambda")
        print("3. curl -XPOST 'http://localhost:9000/2015-03-31/functions/function/invocations' -d '{\"url\": \"https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg\"}'")
    else:
        print("Docker is running. Building image...")
        
        # Build the Docker image
        print("\n[Step 1] Building Docker image...")
        result = subprocess.run(['docker', 'build', '-t', 'hair-classifier-lambda', '.'], 
                              capture_output=True, text=True)
        print(result.stdout)
        if result.stderr:
            print("STDERR:", result.stderr)
        
        if result.returncode == 0:
            print("✓ Image built successfully!")
            
            # Stop any existing container with the same name
            subprocess.run(['docker', 'stop', 'hair-lambda'], capture_output=True)
            subprocess.run(['docker', 'rm', 'hair-lambda'], capture_output=True)
            
            # Run the container in detached mode
            print("\n[Step 2] Starting container...")
            result = subprocess.run(['docker', 'run', '-d', '-p', '9000:8080', 
                                   '--name', 'hair-lambda', 'hair-classifier-lambda'],
                                  capture_output=True, text=True)
            print(result.stdout)
            if result.stderr:
                print("STDERR:", result.stderr)
            
            if result.returncode == 0:
                print("✓ Container started!")
                print("Waiting for container to be ready...")
                time.sleep(3)
                
                # Test the lambda function
                print("\n[Step 3] Testing the lambda function...")
                test_url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
                payload = {"url": test_url}
                
                try:
                    response = requests.post(
                        "http://localhost:9000/2015-03-31/functions/function/invocations",
                        json=payload,
                        timeout=30
                    )
                    
                    if response.status_code == 200:
                        result_data = response.json()
                        model_output = result_data.get('result', result_data)
                        print(f"✓ Request successful!")
                        print(f"\nAnswer to Question 6: Model output is {model_output}")
                        print(f"\nFull response: {json.dumps(result_data, indent=2)}")
                    else:
                        print(f"Error: Status code {response.status_code}")
                        print(f"Response: {response.text}")
                except requests.exceptions.RequestException as e:
                    print(f"Error making request: {e}")
                    print("\nYou can also test manually with:")
                    print(f"curl -XPOST 'http://localhost:9000/2015-03-31/functions/function/invocations' -d '{json.dumps(payload)}'")
        else:
            print("✗ Failed to build image. Check the error messages above.")
            
except subprocess.TimeoutExpired:
    print("ERROR: Docker daemon is not responding!")
    print("Please start Docker Desktop and try again.")
except FileNotFoundError:
    print("ERROR: Docker is not installed!")
    print("Please install Docker Desktop and try again.")
except Exception as e:
    print(f"Error: {e}")


In [2]:
# Question 6: Run the Docker container and test

import subprocess
import json
import time
import requests

# Stop any existing container
subprocess.run(['docker', 'stop', 'hair-lambda'], capture_output=True)
subprocess.run(['docker', 'rm', 'hair-lambda'], capture_output=True)

# Build the Docker image
print("Building Docker image...")
result = subprocess.run(['docker', 'build', '-t', 'hair-classifier-lambda', '.'], 
                      capture_output=True, text=True)
if result.returncode != 0:
    print("Build failed!")
    print(result.stderr)
else:
    print("✓ Image built successfully")
    
    # Run the container
    print("\nStarting container...")
    result = subprocess.run(['docker', 'run', '-d', '-p', '9000:8080', 
                           '--name', 'hair-lambda', 'hair-classifier-lambda'],
                          capture_output=True, text=True)
    if result.returncode == 0:
        print("✓ Container started")
        print("Waiting for container to be ready...")
        time.sleep(3)
        
        # Test the lambda function
        print("\nTesting with image URL...")
        test_url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
        payload = {"url": test_url}
        
        try:
            response = requests.post(
                "http://localhost:9000/2015-03-31/functions/function/invocations",
                json=payload,
                timeout=30
            )
            
            if response.status_code == 200:
                result_data = response.json()
                if isinstance(result_data, dict) and 'body' in result_data:
                    body = json.loads(result_data['body'])
                    model_output = body.get('result', result_data)
                else:
                    model_output = result_data.get('result', result_data)
                
                print(f"✓ Request successful!")
                print(f"\nModel output: {model_output}")
                print(f"\nAnswer to Question 6: {model_output:.2f}")
                
                # Determine which option it matches
                if abs(model_output - (-0.10)) < 0.01:
                    print("\n✓ Matches option: -0.10")
                elif abs(model_output - 0.10) < 0.01:
                    print("\n✓ Matches option: 0.10")
                elif abs(model_output - (-1.0)) < 0.1:
                    print("\n✓ Matches option: -1.0")
                elif abs(model_output - 1.0) < 0.1:
                    print("\n✓ Matches option: 1.0")
            else:
                print(f"Error: Status code {response.status_code}")
                print(f"Response: {response.text}")
        except Exception as e:
            print(f"Error: {e}")
    else:
        print("Failed to start container")
        print(result.stderr)


Building Docker image...
✓ Image built successfully

Starting container...
✓ Container started
Waiting for container to be ready...

Testing with image URL...
✓ Request successful!

Model output: -0.10220833122730255

Answer to Question 6: -0.10

✓ Matches option: -0.10
